In [ ]:
import numpy as np
import pandas as pd
from typing import Tuple
from transformers import pipeline
from huggingface_hub import login
import plotly.express as px
from IPython.display import display
from pathlib import Path

from analysis_utils import (
    combine_transaction_files, 
    split_dataframe_by_search, 
    summarize_search_category,
    process_search_strings, 
    create_sunburst_chart, 
    create_expense_table, 
    display_expense_table, 
    export_expense_table)


c:\Users\mschm\source\BudgetAnalysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print("Downloading/Loading the financial model...")
pipe = pipeline("text-classification", model="kuro-08/bert-transaction-categorization")
category_map = {
    "LABEL_0": "Utilities", "LABEL_1": "Health", "LABEL_2": "Dining",
    "LABEL_3": "Travel", "LABEL_4": "Education", "LABEL_5": "Subscription",
    "LABEL_6": "Family", "LABEL_7": "Food", "LABEL_8": "Festivals",
    "LABEL_9": "Culture", "LABEL_10": "Apparel", "LABEL_11": "Transportation",
    "LABEL_12": "Investment", "LABEL_13": "Shopping", "LABEL_14": "Groceries",
    "LABEL_15": "Documents", "LABEL_16": "Grooming", "LABEL_17": "Entertainment",
    "LABEL_18": "Social Life"
}

Downloading/Loading the financial model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10041.03it/s]


In [9]:
test_items = [
    "Netflix.com", 
    "Whole Foods Market", 
    "Vanguard Investment Group", 
    "SIGNATURE REW+ SQ *ELLA'S ACRES - CORNIN gosq.com NY Date 11/11/25 04692160000058806588060 5411 Card 42 #3444", 
    "SIGNATURE REW+ SQ *DIPPITY DO DAHS Corning NY Date 05/16/25 04692160000054731547310 5814 Card 42 #3444",
    "SIGNATURE REW+ SQ *3 BEARS GLUTEN FREE B Potsdam NY Date 06/28/25 04692160000040039400390 5812 Card 42 #3444",
    "SIGNATURE REW+ SQ *IRON FLAMINGO BREWERY Corning NY Date 03/05/25 04692160000066071660710 5813 Card 42 #3444"]

for item in test_items:
    raw_result = pipe(item)[0]
    raw_label = raw_result['label'] # This is "LABEL_X"
    
    # Use our map to get the real name
    clean_category = category_map.get(raw_label, "Unknown")
    confidence = round(raw_result['score'] * 100, 1)
    
    print(f"Transaction: {item} -> {clean_category} ({confidence}%)")

Transaction: Netflix.com -> Entertainment (65.1%)
Transaction: Whole Foods Market -> Food (57.6%)
Transaction: Vanguard Investment Group -> Investment (99.2%)
Transaction: SIGNATURE REW+ SQ *ELLA'S ACRES - CORNIN gosq.com NY Date 11/11/25 04692160000058806588060 5411 Card 42 #3444 -> Dining (21.7%)
Transaction: SIGNATURE REW+ SQ *DIPPITY DO DAHS Corning NY Date 05/16/25 04692160000054731547310 5814 Card 42 #3444 -> Grooming (75.5%)
Transaction: SIGNATURE REW+ SQ *3 BEARS GLUTEN FREE B Potsdam NY Date 06/28/25 04692160000040039400390 5812 Card 42 #3444 -> Grooming (54.0%)
Transaction: SIGNATURE REW+ SQ *IRON FLAMINGO BREWERY Corning NY Date 03/05/25 04692160000066071660710 5813 Card 42 #3444 -> Subscription (39.5%)


In [ ]:
# login(token=my_token)

In [65]:
# transaction_classifier = pipeline(
#     "text-classification", 
#     model="mitulshah/global-financial-transaction-classifier"
# )

In [2]:
transaction_sheets = {
    "Budget Spreadsheet (Master) - 2025JointVisaTransactions.csv": ["Posting Date", "Amount", "Description"],
    "Budget Spreadsheet (Master) - 2025MikeVisaTransactions.csv": ["Posting Date", "Amount", "Description"],
    "Budget Spreadsheet (Master) - 2025WellsFargoTransactions.csv": ["Date", "Amount", "Description"]
}

In [3]:
df = combine_transaction_files(transaction_sheets, base_path="2025Transactions", parse_dates=True, sort_by_date=False)
print(df.shape)
df.head()

(924, 4)


,Source,Date,Amount,Description
0,Budget Spreadsheet (Master) - 2025JointVisaTra...,2025-12-31,-143.13,SIGNATURE REW+ GRAPES AND GRAINS WIN 607-21509...
1,Budget Spreadsheet (Master) - 2025JointVisaTra...,2025-12-29,-26.64,SIGNATURE REW+ AGT SNACK MART INC ELMIRA NY Da...
2,Budget Spreadsheet (Master) - 2025JointVisaTra...,2025-12-29,-102.97,SIGNATURE REW+ SP BABYLIST CHECKOUT.BABY CA Da...
3,Budget Spreadsheet (Master) - 2025JointVisaTra...,2025-12-29,-1.99,SIGNATURE REW+ WEGMANS #070 ELMIRA NY Date 12/...
4,Budget Spreadsheet (Master) - 2025JointVisaTra...,2025-12-27,-105.00,SIGNATURE REW+ MINDBODY - VIBRANTLIFE MINDBODY...


In [4]:
SEARCH_STRINGS = [
    {"Shopping": ["WEGMANS", "TARGET", "COSTCO"]},
    "AMAZON",
    "Google",
    "CORNING SULLIVAN PARK",
    "CULLIGAN",
    "YMCA Rochester",
    {"Car": ["EZPASS", "E-Z*PASSNY", "GOODYEAR AUTO"]},
    {"Dog": ["VETERINARY", "HAPPY TAILS", "CHEWY.COM", "PETCO"]},
    {"Utilities": ["ELMIRA WATER BOARD", "CASELLA WASTE", "EMPIRE TELEPHONE"]},
    {"Streaming/Entertainment": ["HULUPLUS", "Peacock", "Netflix", "PARAMOUNT+", "MAX.COM", "Audible", "Disney Plus", "Spotify", "STEAMGAMES"]},
    {"Health": ["Schweiger Dermatology", "MARC T RILEY PT", "CVS/PHARMACY", "MICHAEL R RIZZONE"]},
    {"Skiing": ["SKI COMPANY"]},
    {"Running": ["CONFLUENCE"]},
    {"Home": ["LOWES", "HOME DEPOT"]}
]

In [5]:
summed_transactions, remaining_df = process_search_strings(df, SEARCH_STRINGS)
print(remaining_df.shape)
summed_transactions

(356, 4)


{'Shopping': {'WEGMANS': np.float64(6712.979999999999),
  'TARGET': np.float64(593.1099999999999),
  'COSTCO': np.float64(562.79)},
 'AMAZON': np.float64(2954.22),
 'Google': np.float64(185.52999999999997),
 'CORNING SULLIVAN PARK': np.float64(1155.5199999999998),
 'CULLIGAN': np.float64(537.25),
 'YMCA Rochester': np.float64(1377.6000000000001),
 'Car': {'EZPASS': np.float64(108.75),
  'E-Z*PASSNY': np.float64(-0.0),
  'GOODYEAR AUTO': np.float64(1166.83)},
 'Dog': {'VETERINARY': np.float64(1260.15),
  'HAPPY TAILS': np.float64(5462.639999999999),
  'CHEWY.COM': np.float64(60.77),
  'PETCO': np.float64(4.54)},
 'Utilities': {'ELMIRA WATER BOARD': np.float64(475.93999999999994),
  'CASELLA WASTE': np.float64(581.7599999999999),
  'EMPIRE TELEPHONE': np.float64(675.0)},
 'Streaming/Entertainment': {'HULUPLUS': np.float64(2.0),
  'Peacock': np.float64(182.88000000000002),
  'Netflix': np.float64(188.9),
  'PARAMOUNT+': np.float64(90.92999999999999),
  'MAX.COM': np.float64(69.93),
  'Aud

In [6]:
fig = create_sunburst_chart(summed_transactions)
fig.show()

In [58]:
summary_df = display_expense_table(summed_transactions)

Category,Subcategory,Amount
Shopping,WEGMANS,"$6,712.98"
Shopping,TARGET,$593.11
Shopping,COSTCO,$562.79
Shopping,TOTAL,"$7,868.88"
,,
AMAZON,—,"$2,954.22"
Google,—,$185.53
CORNING SULLIVAN PARK,—,"$1,155.52"
CULLIGAN,—,$537.25
YMCA Rochester,—,"$1,377.60"


In [ ]:
# exported_files = export_expense_table(
#     summed_transactions,
#     filename='2025_expenses_summary',
#     formats=['csv'])

✓ Exported to 2025_expenses_summary.csv


In [59]:
remaining_df["Description"].unique().tolist()

['SIGNATURE REW+ GRAPES AND GRAINS WIN 607-2150923 NY Date 12/30/25 04045470000053733537330 5921 Card 42 #5856',
 'SIGNATURE REW+ AGT SNACK MART INC ELMIRA NY Date 12/29/25 04034540000072132721320 5542 Card 42 #5856',
 'SIGNATURE REW+ SP BABYLIST CHECKOUT.BABY CA Date 12/29/25 04011340000013213132130 5999 Card 42 #3444',
 'SIGNATURE REW+ MINDBODY - VIBRANTLIFE MINDBODYONLIN CA Date 12/27/25 04492160000070023700230 7997 Card 42 #5856',
 "SIGNATURE REW+ SMILEY'S QUICK STOP #1 ELMIRA NY Date 12/26/25 04801970000066717667170 5542 Card 42 #5856",
 'SIGNATURE REW+ SQ *TRANQUIL MIND COUNSEL gosq.com NY Date 12/23/25 04692160000039272392720 8099 Card 42 #5856',
 'From Share 0009',
 'SIGNATURE REW+ MERRELL.COM 800-288-3124 MI Date 12/16/25 04431050000031252312520 5661 Card 42 #5856',
 'SIGNATURE REW+ STARBUCKS 8007827282 800-782-7282 WA Date 12/13/25 04692160000006285062850 5814 Card 42 #5856',
 'SIGNATURE REW+ SPEEDWAY 44934 PENFIELD NY Date 12/12/25 04137460000026757267570 5542 Card 42 #3444'